# NoFluffJobs API search

Query the NoFluffJobs posting search endpoint, inspect the response, and collect the returned postings into a clean list.

## 1. Imports

We use `httpx` for the HTTP request and `pandas` later for tabular display.

In [31]:
import httpx
import pandas as pd

## 2. Endpoint and query parameters

`params` are sent as query string arguments. Salary is requested in EUR per month. The `region`/`requirement` filters are left commented so the search stays broad.

In [32]:
url = "https://nofluffjobs.com/api/search/posting"

# Query parameters sent on the URL
params = {
    "salaryCurrency": "EUR",
    "salaryPeriod": "month",
    # "region": "nl",          # 'nl', 'pl', 'cz', 'hu', etc.
    "language": "en",
    "page": 1,
}

## 3. Request body

The backend expects the key `criteriaSearch` (not `criteria`). Add filters such as `requirement` inside it to narrow the search.

In [33]:
payload = {
    "criteriaSearch": {
        # "requirement": ["python"]
    }
}

## 4. HTTP headers

A browser-like `User-Agent` plus `Referer`/`Origin` are required for the API to accept the request.

In [34]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Referer": "https://nofluffjobs.com/",
    "Origin": "https://nofluffjobs.com",
}

## 5. Send the request

In [35]:
response = httpx.post(url, params=params, json=payload, headers=headers)
print(f"HTTP status: {response.status_code}")

HTTP status: 200


## 6. Parse and clean the response

On success, extract the `postings` list. Each posting is kept as-is for now; `salary` is read out to confirm it is present.

In [36]:
cleaned_postings = []

if response.status_code == 200:
    data = response.json()
    postings = data.get("postings", [])
    print(f"Success! Retrieved {len(postings)} postings.")

    for job in postings:
        salary = job.get("salary", {})
        cleaned_postings.append(job)
else:
    print(f"Failed with status {response.status_code}: {response.text[:200]}")

Success! Retrieved 214 postings.


## 7. Inspect the collected postings

In [37]:
display(cleaned_postings)

[{'id': 'senior-data-engineer-alterdata-io-Remote-1',
  'name': 'Alterdata.io sp. z o.o.',
  'location': {'places': [{'city': 'Remote',
     'url': 'senior-data-engineer-alterdata-io-remote-1'},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lower-silesian',
     'url': 'senior-data-engineer-alterdata-io-lower-silesian-1',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'kuyavian-pomeranian',
     'url': 'senior-data-engineer-alterdata-io-kuyavian-pomeranian-1',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lodz',
     'url': 'senior-data-engineer-alterdata-io-lodz-1',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lublin',
     'url': 'senior-data-engineer-alterdata-io-lublin-1',
     'provinceOnly': True},
    {'country': {'code': 'POL', 'name': 'Poland'},
     'province': 'lubusz',
     'url': 'senior-data-engineer

### Optional: view as a DataFrame

Flatten a few key fields for easier inspection.

In [38]:
if cleaned_postings:
    df = pd.DataFrame([
        {
            "title": j.get("title"),
            "company": j.get("name"),
            "salary_from": j.get("salary", {}).get("from"),
            "salary_to": j.get("salary", {}).get("to"),
            "currency": j.get("salary", {}).get("currency"),
            "remote": j.get("fullyRemote"),
        }
        for j in cleaned_postings
    ])
    display(df)
else:
    print("No postings collected.")

,title,company,salary_from,salary_to,currency,remote
0,Senior Data Engineer,Alterdata.io sp. z o.o.,4368.0,6048.0,EUR,False
1,Senior Data Engineer,Alterdata.io sp. z o.o.,4368.0,6048.0,EUR,False
2,Senior Data Engineer,Alterdata.io sp. z o.o.,4368.0,6048.0,EUR,False
3,Senior Data Engineer,Alterdata.io sp. z o.o.,4368.0,6048.0,EUR,False
4,Senior Data Engineer,Alterdata.io sp. z o.o.,4368.0,6048.0,EUR,False
...,...,...,...,...,...,...
209,Senior Java Full Stack Engineer,1dea,5208.0,6048.0,EUR,False
210,Senior Java Full Stack Engineer,1dea,5208.0,6048.0,EUR,False
211,Senior Java Full Stack Engineer,1dea,5208.0,6048.0,EUR,False
212,Lead IT Expert,Roche,4329.0,8040.0,EUR,False


## 8. Persist to SQLite

Store each posting in a local SQLite database.

In [40]:
import sqlite3
import json as _json

db_path = "nofluff_jobs.db"
conn = sqlite3.connect(db_path)

# Flatten every field of each posting into columns, keeping all of them
df_all = pd.json_normalize(cleaned_postings)
if "id" in df_all.columns:
    df_all = df_all.set_index("id")

# Make nested values (lists/dicts) storable in SQLite
df_all = df_all.map(lambda x: _json.dumps(x) if isinstance(x, (list, dict)) else x)

df_all.to_sql("postings", conn, if_exists="replace", index=True, index_label="id")
conn.commit()

print(f"Saved {len(df_all)} postings with {len(df_all.columns)} columns to {db_path}")

cur = conn.cursor()
print(cur.execute("SELECT COUNT(*) FROM postings").fetchone())
print([r[1] for r in cur.execute("PRAGMA table_info(postings)").fetchall()][:10])
conn.close()


Saved 214 postings with 51 columns to nofluff_jobs.db
(214,)
['id', 'name', 'posted', 'title', 'technology', 'category', 'seniority', 'url', 'regions', 'fullyRemote']
